In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets
from torch.utils.data import DataLoader, Subset
import numpy as np
import os

# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Load the pre-trained weights and model
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# Strip the final classification layer
model.fc = nn.Identity()

# Move to GPU/CPU and set to evaluation mode (CRITICAL)
model = model.to(device)
model.eval()

print("Model loaded and classification head removed.")

Model loaded and classification head removed.


In [6]:
# Use the exact transforms expected by ResNet50
preprocess = weights.transforms()

dataset_path = 'train'
full_dataset = datasets.ImageFolder(root=dataset_path, transform=preprocess)

# CRITICAL CHANGE: We removed the Subset. We are loading the full dataset.
dataloader = DataLoader(full_dataset, batch_size=32, shuffle=False)

class_names = full_dataset.classes
print(f"Loaded {len(class_names)} species classes.")
print(f"Total images to process: {len(full_dataset)}")

Loaded 100 species classes.
Total images to process: 12594


In [7]:
all_embeddings = []
all_labels = []

print(f"Starting extraction for {len(subset)} images...")

# Disable gradients to save memory
with torch.no_grad():
    for images, labels in dataloader:
        images = images.to(device)
        
        # Forward pass to get [batch_size, 2048] features
        features = model(images) 
        
        # Move back to CPU and convert to numpy
        all_embeddings.append(features.cpu().numpy())
        all_labels.extend(labels.numpy())

# Stack into contiguous numpy arrays
embeddings_matrix = np.vstack(all_embeddings)
labels_array = np.array(all_labels)

print(f"Extraction complete! Shape: {embeddings_matrix.shape}")

Starting extraction for 500 images...
Extraction complete! Shape: (12594, 2048)


In [9]:
os.makedirs('processed', exist_ok=True)

# Notice we changed the filenames to "_full" instead of "_500"
np.save('processed/embeddings_full.npy', embeddings_matrix)
np.save('processed/labels_full.npy', labels_array)

class_mapping = {i: name for i, name in enumerate(class_names)}
np.save('processed/class_mapping.npy', class_mapping)

print("SUCCESS: Full dataset saved to the /processed/ folder!")

SUCCESS: Full dataset saved to the /processed/ folder!
